In [2]:
! pip install tqdm

     ---------------------------------------- 78.4/78.4 kB 4.5 MB/s eta 0:00:00


In [3]:
import os
import numpy as np
import glob
import csv
from tqdm import tqdm

# Configuration
transforms_dir = "./sample_dataset/transforms"
output_csv = "./sample_dataset/metadata.csv"

# The exact lists from your original collection code
distances = [5, 7, 10]
pitches = [10, 30, 45]
yaws = list(range(0, 360, 30))

def get_sequence_index(dist, pitch, yaw):
    """Maps the camera parameters to a strict integer for sequence tracking."""
    try:
        d_idx = distances.index(int(dist))
        p_idx = pitches.index(int(pitch))
        y_idx = yaws.index(int(yaw))
    except ValueError:
        return -1 
    
    return d_idx * (len(pitches) * len(yaws)) + p_idx * len(yaws) + y_idx

def main():
    npy_files = sorted(glob.glob(os.path.join(transforms_dir, "*.npy")))
    print(f"Found {len(npy_files)} transform files to process.")

    current_vehicle = None
    prev_seq_index = -1
    sampleset_no = 0 
    location_id = 0
    
    # Prepare the data list with a header row
    csv_data = [["filename", "distance", "pitch", "yaw", "vehicle_id", "sampleset_no", "location_id"]]

    for file in tqdm(npy_files, desc="Extracting Metadata"):
        try:
            # Read the original file (READ-ONLY)
            data = np.load(file)
            dist, pitch, yaw, vehicle = data[0], data[1], data[2], data[3]
            
            # Extract just the filename (e.g., '00000.npy' -> '00000')
            base_filename = os.path.splitext(os.path.basename(file))[0]
            
            seq_index = get_sequence_index(dist, pitch, yaw)
            
            if vehicle != current_vehicle:
                current_vehicle = vehicle
                sampleset_no += 1
                location_id = 1
            elif seq_index <= prev_seq_index:
                sampleset_no += 1
                location_id += 1
            
            prev_seq_index = seq_index
            
            # Append to our list instead of saving to the .npy file
            csv_data.append([base_filename, dist, pitch, yaw, vehicle, sampleset_no, location_id])
            
        except Exception as e:
            print(f"Error reading {file}: {e}")

    # Write everything to a brand new CSV file
    print(f"\nWriting to {output_csv}...")
    with open(output_csv, mode='w', newline='') as f:
        writer = csv.writer(f)
        writer.writerows(csv_data)

    print("✅ Metadata extraction complete! Your original data is untouched.")

if __name__ == '__main__':
    main()

Found 24083 transform files to process.


Extracting Metadata: 100%|██████████| 24083/24083 [01:52<00:00, 214.27it/s]



Writing to ./sample_dataset/metadata.csv...
✅ Metadata extraction complete! Your original data is untouched.
